# E2B Example: Preview Environment Builder

This is the smallest showcase in the set: one E2B sandbox agent inspects a tiny app workspace, starts a local server, and returns a preview URL plus a concise summary.

In [ ]:
import importlib
import os
import subprocess
import sys
from pathlib import Path

repo_root = Path.cwd()
if not (repo_root / "src" / "agents").exists():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src" / "agents").exists():
            repo_root = candidate
            break
        if (candidate / "openai-agents-python-preview" / "src" / "agents").exists():
            repo_root = candidate / "openai-agents-python-preview"
            break

src_root = repo_root / "src"
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

env_path = None
for candidate in [Path.cwd(), *Path.cwd().parents, repo_root, *repo_root.parents]:
    maybe_env = candidate / ".env"
    if maybe_env.exists():
        env_path = maybe_env
        break

if env_path is not None:
    try:
        from dotenv import load_dotenv

        load_dotenv(env_path, override=False)
        env_loader = "python-dotenv"
    except Exception:
        for raw_line in env_path.read_text().splitlines():
            line = raw_line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))
        env_loader = "manual-fallback"
else:
    env_loader = "not-found"

for module_name in ["agents", "openai", "e2b", "e2b_code_interpreter"]:
    try:
        importlib.import_module(module_name)
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", f"{repo_root}[e2b]"])
        break

print(f"Using repo root: {repo_root}")
print(f".env loader: {env_loader} ({env_path})")
print(f"OPENAI_API_KEY set: {bool(os.getenv('OPENAI_API_KEY'))}")
print(f"E2B_API_KEY set: {bool(os.getenv('E2B_API_KEY'))}")

In [ ]:
from pydantic import BaseModel, Field

from agents import ModelSettings, Runner
from agents.extensions.sandbox import (
    E2BSandboxClient,
    E2BSandboxClientOptions,
    E2BSandboxType,
)
from agents.run import RunConfig
from agents.sandbox import SandboxAgent, SandboxRunConfig
from examples.sandbox.misc.example_support import text_manifest
from examples.sandbox.misc.workspace_shell import WorkspaceShellCapability

MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4")
SANDBOX_TYPE = E2BSandboxType(os.getenv("E2B_SANDBOX_TYPE", E2BSandboxType.E2B.value))
TEMPLATE = os.getenv("E2B_TEMPLATE") or None


class PreviewResult(BaseModel):
    summary: str
    app_port: int = Field(description="The app port to expose.")
    evidence_files: list[str] = Field(min_length=1)


manifest = text_manifest(
    {
        "index.html": (
            "<html><body><h1>Hello from E2B</h1><p>Preview environment is live.</p></body></html>"
        ),
        "README.md": "Serve index.html on port 8765.\n",
    }
)

agent = SandboxAgent(
    name="Preview Builder",
    model=MODEL,
    instructions=(
        "Inspect the tiny app workspace, start a preview server, and summarize what is running."
    ),
    developer_instructions=(
        "Use the shell tool. Start a background HTTP server with "
        "python -m http.server 8765. app_port must be 8765."
    ),
    default_manifest=manifest,
    capabilities=[WorkspaceShellCapability()],
    model_settings=ModelSettings(tool_choice="required"),
    output_type=PreviewResult,
)

client = E2BSandboxClient()
session = await client.create(
    manifest=manifest,
    options=E2BSandboxClientOptions(
        sandbox_type=SANDBOX_TYPE,
        template=TEMPLATE,
        timeout=300,
        exposed_ports=(8765,),
    ),
)
await session.start()

run_config = RunConfig(sandbox=SandboxRunConfig(session=session))

result = await Runner.run(agent, "Build and expose the preview environment.", run_config=run_config)  # type: ignore[top-level-await]  # noqa: F704
endpoint = await session.resolve_exposed_port(8765)
preview_url = endpoint.url_for("http")

print(result.final_output)
print(preview_url)